# Kapitel 3: Wörter werden zu Zahlen

## Bisher...

Wir haben 400.000 Bewertungen geladen (Kapitel 1) und bereinigt (Kapitel 2).
Die Texte sind sauber, normalisiert und haben ein klares Sentiment-Label.

## Das Problem

Eine Maschine kann nicht lesen. Sie versteht weder *"great"* noch *"terrible"*.
Was sie versteht, sind **Zahlen**. Wir müssen also einen Weg finden,
jeden Text in einen numerischen Vektor zu verwandeln — und zwar so,
dass die Bedeutung erhalten bleibt.

## Die Lösung: TF-IDF

**TF-IDF** (Term Frequency – Inverse Document Frequency) misst die Relevanz
eines Wortes. Die Idee:

- Ein Wort, das in *einem* Review oft vorkommt, ist **für dieses Review wichtig** (TF)
- Ein Wort, das in *allen* Reviews vorkommt, ist **nicht aussagekräftig** (IDF senkt es)
- **TF × IDF** = Wie einzigartig-wichtig ist dieses Wort für dieses Dokument?

So wird *"great"* in einem positiven Review hochgewichtet,
während *"the"* überall vorkommt und heruntergewichtet wird.

## 3.1 Bereinigte Daten laden

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews – Tokenisierung & TF-IDF") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

df = spark.read.parquet(
    "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/cleaned_reviews.parquet"
)

print(f"Daten geladen: {df.count():,} Zeilen")
df.printSchema()
df.show(3, truncate=80)

## 3.2 Schritt 1: Tokenisierung

Der erste Schritt: Jeden Text in eine **Liste einzelner Wörter** zerlegen.

*"this book is great"* → `["this", "book", "is", "great"]`

Spark MLlib bietet dafür den `Tokenizer`, der am Leerzeichen trennt.

In [ ]:
from pyspark.ml.feature import Tokenizer

tokenizer = Tokenizer(inputCol="text", outputCol="tokens")
df_tokens = tokenizer.transform(df)

df_tokens.select("text", "tokens").show(3, truncate=80)

In [ ]:
from pyspark.sql.functions import size, avg, min as spark_min, max as spark_max, round as spark_round

df_tokens.withColumn("token_count", size("tokens")).select(
    spark_min("token_count").alias("Min"),
    spark_round(avg("token_count"), 0).alias("Durchschnitt"),
    spark_max("token_count").alias("Max")
).show()

## 3.3 Schritt 2: StopWords entfernen

Nicht alle Wörter tragen Bedeutung. *"the"*, *"is"*, *"at"*, *"which"* —
diese sogenannten **StopWords** kommen in jedem Text vor und verwässern die Analyse.

Wir entfernen sie, damit nur die **bedeutungstragenden Wörter** übrig bleiben:

*`["this", "book", "is", "great"]`* → *`["book", "great"]`*

In [ ]:
from pyspark.ml.feature import StopWordsRemover

remover = StopWordsRemover(inputCol="tokens", outputCol="filtered_tokens")
df_filtered = remover.transform(df_tokens)

df_filtered.select("tokens", "filtered_tokens").show(3, truncate=80)

In [ ]:
from pyspark.sql.functions import col

df_compare = df_filtered \
    .withColumn("before", size("tokens")) \
    .withColumn("after", size("filtered_tokens")) \
    .withColumn("removed", col("before") - col("after"))

df_compare.select(
    spark_round(avg("before"), 1).alias("Tokens vorher"),
    spark_round(avg("after"), 1).alias("Tokens nachher"),
    spark_round(avg("removed"), 1).alias("Entfernt (Durchschnitt)")
).show()

## 3.4 Welche Wörter bleiben übrig?

Eine spannende Frage: **Unterscheiden sich die häufigsten Wörter zwischen
positiven und negativen Bewertungen?**

Wenn ja, hat unser Modell eine Chance, die Stimmung zu erkennen.

In [ ]:
from pyspark.sql.functions import explode, count as spark_count

df_words = df_filtered.select("label", explode("filtered_tokens").alias("word"))

print("=== Top 15 Wörter – POSITIV (label=1) ===")
df_words.filter(col("label") == 1) \
    .groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).show(15)

print("=== Top 15 Wörter – NEGATIV (label=0) ===")
df_words.filter(col("label") == 0) \
    .groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).show(15)

### Was fällt auf?

Einige Wörter wie *book* und *one* tauchen in beiden Listen auf — logisch,
denn alle sprechen über dieselben Produkte. Aber die **feinen Unterschiede** sind vielsagend:

- **Positiv:** *great, love, best, well* → Wörter der Begeisterung
- **Negativ:** *money, bought, buy, product* → Wörter der Enttäuschung und des Kaufs

TF-IDF wird diese Unterschiede mathematisch hervorheben — gemeinsame Wörter
bekommen niedrige IDF-Werte, während die unterscheidenden Wörter hohe Werte erhalten.

## 3.5 Schritt 3: TF-IDF berechnen

Jetzt kommt der mathematische Kern: Wir verwandeln jede Wortliste
in einen **10.000-dimensionalen Vektor**.

- `HashingTF` zählt, wie oft jedes Wort in einem Review vorkommt (TF)
- `IDF` gewichtet diese Zählungen nach ihrer Seltenheit im gesamten Datensatz

In [ ]:
from pyspark.ml.feature import HashingTF, IDF

hashing_tf = HashingTF(
    inputCol="filtered_tokens",
    outputCol="raw_features",
    numFeatures=10000
)
df_tf = hashing_tf.transform(df_filtered)

print("HashingTF abgeschlossen.")
df_tf.select("filtered_tokens", "raw_features").show(2, truncate=80)

In [ ]:
idf = IDF(inputCol="raw_features", outputCol="tfidf_features")
idf_model = idf.fit(df_tf)
df_tfidf = idf_model.transform(df_tf)

print("IDF-Berechnung abgeschlossen.")
df_tfidf.select("label", "filtered_tokens", "tfidf_features").show(2, truncate=80)

## 3.6 Was steckt in einem TF-IDF-Vektor?

Jedes Review ist jetzt ein **Sparse Vector** — ein Vektor mit 10.000 Dimensionen,
von denen die meisten null sind. Nur die Dimensionen, die zu Wörtern im Text gehören,
haben einen Wert.

In [ ]:
sample = df_tfidf.select("tfidf_features").first()[0]

print(f"Vektor-Typ:   {type(sample).__name__}")
print(f"Dimensionen:  {sample.size}")
print(f"Nicht-Null:   {len(sample.indices)} Einträge")
print(f"\nErste 10 Indizes:  {sample.indices[:10]}")
print(f"Erste 10 Werte:    {[round(v, 4) for v in sample.values[:10]]}")

### Was bedeutet das?

Dieses Review hat ca. 30 verschiedene Wörter nach StopWord-Entfernung.
Jedes Wort wurde auf eine der 10.000 Dimensionen abgebildet.
Die Werte zeigen: Je höher der TF-IDF-Wert, desto einzigartiger und wichtiger
ist dieses Wort für genau dieses Review.

## 3.7 Ergebnisse als Parquet speichern

In [ ]:
df_output = df_tfidf.select("label", "text", "filtered_tokens", "tfidf_features")

output_path = "/Users/alperbildiren/PYSPARK_AMAZON_PROJECT/output/tfidf_features.parquet"
df_output.write.parquet(output_path, mode="overwrite")
print(f"TF-IDF-Ergebnisse gespeichert: {output_path}")

In [ ]:
df_check = spark.read.parquet(output_path)
print(f"Parquet gelesen: {df_check.count():,} Zeilen, {len(df_check.columns)} Spalten")
df_check.printSchema()

## 3.8 Visualisierung: Die Sprache der Emotionen

In [ ]:
import matplotlib.pyplot as plt

top_pos = df_words.filter(col("label") == 1) \
    .groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).limit(15).toPandas()

top_neg = df_words.filter(col("label") == 0) \
    .groupBy("word").agg(spark_count("*").alias("count")) \
    .orderBy(col("count").desc()).limit(15).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(top_pos["word"][::-1], top_pos["count"][::-1], color="#5DCAA5", edgecolor="#0F6E56")
axes[0].set_title("Top 15 Wörter – POSITIV")
axes[0].set_xlabel("Häufigkeit")

axes[1].barh(top_neg["word"][::-1], top_neg["count"][::-1], color="#F09595", edgecolor="#A32D2D")
axes[1].set_title("Top 15 Wörter – NEGATIV")
axes[1].set_xlabel("Häufigkeit")

plt.tight_layout()
plt.show()

## Kapitel 3 — Zusammenfassung

| Schritt | Werkzeug | Was passiert? |
|---------|----------|---------------|
| Tokenisierung | `Tokenizer` | Text → Wortliste |
| StopWords | `StopWordsRemover` | Bedeutungslose Wörter entfernen |
| TF | `HashingTF(10000)` | Wörter → Häufigkeits-Vektoren |
| IDF | `IDF` | Häufigkeiten → Relevanz-Gewichtung |

**Ergebnis:** Jedes Review ist jetzt ein 10.000-dimensionaler Vektor.
Die Wörter *"great"* und *"love"* haben hohe TF-IDF-Werte in positiven Reviews;
*"money"* und *"bought"* in negativen. Die Maschine hat jetzt Material zum Lernen.

**Nächstes Kapitel:** Wir trainieren ein Machine-Learning-Modell, das aus diesen
Vektoren vorhersagt: Ist diese Bewertung positiv oder negativ?